# 3. Persistent Data Ingestion with SQLite Search

This notebook separates **ingestion** from **querying**. We fetch the FAQ data once, write it to a persistent SQLite FTS5 index, and then reopen that index whenever we need to search or run RAG.

## Learning goals

By the end, you should understand:

- how to load and filter the FAQ dataset;
- how `sqlitesearch` persists a full-text index in `faq-search.db`;
- how another process can query the index without downloading the source data again;
- why SQLite creates `-wal` and `-shm` companion files;
- how to reuse the same `RAGBase` pipeline with a different search backend.

The notebook continues from [02-reusable-rag-pipeline.ipynb](02-reusable-rag-pipeline.ipynb) and prepares the persistent retrieval backend used by later RAG and agent workflows.

In [8]:
from ingestion import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1401 documents


In [9]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 139 documents


## 1. Ingest and persist the FAQ documents

This cell downloads the FAQ data, keeps only the LLM Zoomcamp records, and writes each document into the SQLite-backed full-text index. The small delay makes the ingestion process visible and gives another reader time to observe the index growing.

### Why `faq-search.db-wal` and `faq-search.db-shm` appear

This database is using SQLite **WAL mode**. The companion files are created when the database is opened for writing and SQLite needs to coordinate transactions:

- `faq-search.db-wal` is the **write-ahead log**. New committed changes are written here first, allowing readers to continue using the main database while ingestion is in progress. SQLite later checkpoints those changes back into `faq-search.db`.
- `faq-search.db-shm` is the **shared-memory file** used by SQLite connections to coordinate readers, writers, and the WAL file.

They can appear while the ingestion or query connection is open. After a clean close and checkpoint they may disappear, but they can remain when a connection is still active, a process is interrupted, or SQLite has not yet checkpointed the WAL. They are normal SQLite runtime files, not additional datasets, and should not be opened or edited manually.

In [10]:
import time
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq-search.db"
)

for doc in docs_llm:
    index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq-search.db")

Added: I just discovered the course. Can I still join?...
Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: How should I start the course and follow the weekly workflow...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: Where can I track the LLM Zoomcamp syllabus, deadlines, home...
Added: Are there live sessions or office hours for each module?...
Added: Can I use Bluesky for learning in public credits?...
Added: Where is the LLM Zoomcamp Telegram channel?...
Added: Why is the number of documents in the FAQ dataset different ...
Added: The homework submission f

## 2. Reconnect to the persistent index

The ingestion step has already created `faq-search.db`. Here we open the same file again instead of fetching and indexing the FAQ data from scratch.

This is the key difference from an in-memory `minsearch` index: the search data survives when the ingestion process or notebook kernel stops.

In [11]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq-search.db"
)

In [14]:
sqlite_index.count()

242

## 3. Check the index and search the FAQ data

First, count the indexed documents to confirm that the persistent index is available. Then run a full-text search and inspect the questions returned by the index.

The query code does not need to know how the index was built. It only connects to the database and uses the search API.

In [16]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=8)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'Can I submit homework after the deadline, or get a deadline extension?',
 'Can I submit homework after the deadline, or get a deadline extension?']

In [17]:
sqlite_index.search("How do I join the course?")

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '85384a18e5',
  'course': 

## 4. Reuse the RAG pipeline with SQLite

The retrieval backend has changed from in-memory `minsearch` to persistent `sqlitesearch`, but the RAG pipeline remains the same. We only pass the new index into `RAGBase`; retrieval, prompt construction, and answer generation continue to use the same interface.

In [18]:
from rag_pipeline import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [19]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after the course has started. If you want a certificate, you’ll need to submit your project while submissions are still being accepted.


## 5. Compare the two architectures and clean up

With `minsearch`, startup does everything in one process:

```text
fetch data -> build in-memory index -> answer questions
```

With `sqlitesearch`, the responsibilities are separated:

```text
ingestion process -> write faq-search.db
query process -> open faq-search.db -> search and answer
```

This persistent approach avoids repeating slow ingestion after every restart. Close the database connection when you finish so SQLite can release its locks and complete any needed WAL checkpoint.

In [20]:
sqlite_index.close()

After closing the connection, you can delete `faq-search.db` and its companion files if you want to start fresh.    